In [1]:
# =====================================
# 1. Import Libraries
# =====================================
import numpy as np
import pandas as pd
import string

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.utils import to_categorical

In [2]:
# =====================================
# 2. Load Dataset
# =====================================
df = pd.read_csv("/content/qoute_dataset.csv")

quotes = df['quote']

In [3]:
# =====================================
# 3. Text Cleaning
# =====================================
quotes = quotes.str.lower()

translator = str.maketrans('', '', string.punctuation)

quotes = quotes.apply(lambda x: x.translate(translator))

In [4]:
# =====================================
# 4. Tokenization
# =====================================
vocab_size = 2000

tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(quotes)

word_index = tokenizer.word_index

In [5]:
# =====================================
# 5. Convert Text → Sequences
# =====================================
sequences = tokenizer.texts_to_sequences(quotes)

X = []
y = []

for seq in sequences:

    for i in range(1, len(seq)):

        input_seq = seq[:i]
        output_word = seq[i]

        X.append(input_seq)
        y.append(output_word)

In [6]:
# =====================================
# 6. Padding
# =====================================
max_len = 20

X = pad_sequences(X, maxlen=max_len, padding='pre')

y = np.array(y)

In [7]:
# =====================================
# 7. One Hot Encoding
# =====================================
y = to_categorical(y, num_classes=vocab_size)

In [8]:
# =====================================
# 8. Build Model
# =====================================
model = Sequential()

model.add(Embedding(vocab_size, 150))

model.add(Bidirectional(LSTM(150)))

model.add(Dropout(0.2))

model.add(Dense(vocab_size, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# =====================================
# 9. Train Model
# =====================================
history = model.fit(
    X,
    y,
    epochs=60,
    batch_size=128,
    validation_split=0.1
)

Epoch 1/60
536/536 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - accuracy: 0.0505 - loss: 5.9056 - val_accuracy: 0.0706 - val_loss: 5.6981
Epoch 2/60
536/536 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.0986 - loss: 5.4403 - val_accuracy: 0.1108 - val_loss: 5.3706
Epoch 3/60
536/536 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1200 - loss: 5.1751 - val_accuracy: 0.1215 - val_loss: 5.2685
Epoch 4/60
536/536 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.1337 - loss: 4.9948 - val_accuracy: 0.1296 - val_loss: 5.2202
Epoch 5/60
536/536 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1442 - loss: 4.8395 - val_accuracy: 0.1350 - val_loss: 5.1952
Epoch 6/60
536/536 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.1530 - loss: 4.6879 - val_accuracy: 0.1341 - val_loss: 5.1898
Epoch 7/60
536/536 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.1637 - loss: 4.5449 - val_accuracy: 0.1350 - val_loss: 5.2041
Epoch 8/60
536/536 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1724 - loss: 4.3990 - val_a

In [10]:
# =====================================
# 10. Save Model
# =====================================
model.save("next_word_model.keras")

In [11]:
# =====================================
# 11. Index → Word Mapping
# =====================================
index_to_word = {}

for word, index in word_index.items():

    index_to_word[index] = word

In [12]:
# =====================================
# 12. Predict Next Word (Top-K Sampling)
# =====================================
def predict_next_word(text):

    text = text.lower()

    seq = tokenizer.texts_to_sequences([text])[0]

    seq = pad_sequences([seq], maxlen=max_len, padding='pre')

    preds = model.predict(seq, verbose=0)[0]

    top_indices = preds.argsort()[-5:][::-1]

    next_index = np.random.choice(top_indices)

    return index_to_word.get(next_index, "")

In [13]:
# =====================================
# 13. Generate Sentence
# =====================================
def generate_text(seed_text, n_words):

    for i in range(n_words):

        next_word = predict_next_word(seed_text)

        if next_word == "":
            break

        seed_text += " " + next_word

    return seed_text

In [14]:
# =====================================
# 14. Test Prediction
# =====================================
seed = "are you a"

result = generate_text(seed, 10)

print(result)

are you a boy i loved the universe as one has no one


In [15]:
seed = "are you a"

next_word = predict_next_word(seed)

print(next_word)

kind


In [16]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [17]:
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)